# Coastal flood step 06: expected annual damage maps (maximum_scenario)

- Loads asset-level EAD output from step 05.
- Builds geospatial map layers and creates expected annual damage maps.


In [ ]:
from pathlib import Path

import pandas
import numpy
import geopandas
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from mpl_toolkits.axes_grid1 import make_axes_locatable

pandas.set_option('display.max_columns', 200)
pandas.set_option('display.width', 200)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)



In [ ]:
# Core paths and shared map settings
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
results_path = base_path / 'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario'
intersections_path = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections'
direct_damages_path = results_path / 'direct_damages'
damage_estimates_path = results_path / 'damage_estimates'
map_out_dir = damage_estimates_path / 'maps_avoided_ead'
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'

jamaica_metric_grid_crs = 'EPSG:3448'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
sector_order = ['buildings', 'energy', 'transport', 'water']
avoided_ead_column = 'Avoided_EAD_USD'
avoided_ead_colormap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)

if not direct_damages_path.exists():
    raise FileNotFoundError(f'Missing folder: {direct_damages_path}')
if not network_csv.exists():
    raise FileNotFoundError(f'Missing file: {network_csv}')
if not intersections_path.exists():
    raise FileNotFoundError(f'Missing folder: {intersections_path}')
if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')

map_out_dir.mkdir(parents=True, exist_ok=True)
jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)

print(f'direct_damages_path: {direct_damages_path}')
print(f'network_csv: {network_csv}')
print(f'intersections_path: {intersections_path}')
print(f'map_out_dir: {map_out_dir}')



In [ ]:
# Read network metadata and build expected file mapping (asset/layer/id column)
network_details = pandas.read_csv(network_csv)
required_cols = ['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']
missing_cols = [c for c in required_cols if c not in network_details.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in network csv: {missing_cols}')

network_details = network_details[required_cols].drop_duplicates().copy()
network_details['folder_name'] = network_details['asset_gpkg'] + '_' + network_details['asset_layer']
network_details['expected_parquet'] = network_details['folder_name'].apply(
    lambda folder: direct_damages_path / folder / f'{folder}_direct_damages_parameter_set_0.parquet'
)
network_details['exists'] = network_details['expected_parquet'].apply(lambda p: p.exists())

display(network_details[['sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column', 'exists']].sort_values(['sector', 'asset_gpkg', 'asset_layer']))

missing_files = network_details.loc[~network_details['exists'], ['asset_gpkg', 'asset_layer', 'expected_parquet']]
if len(missing_files) > 0:
    print('Missing expected files:')
    display(missing_files)
else:
    print('All expected direct-damage files are present.')


In [ ]:
# Load asset-level EAD output from step 05
asset_out = damage_estimates_path / 'coastal_ead_asset_level_usd.csv'
if not asset_out.exists():
    raise FileNotFoundError(f'Missing asset-level EAD CSV: {asset_out}. Run step 05 calculations notebook first.')
asset_ead = pandas.read_csv(asset_out)
print(f'Loaded asset EAD rows: {len(asset_ead):,} from {asset_out}')
display(asset_ead.head(10))



In [ ]:
# Build geospatial layers for mapping sector avoided EAD using coastal split geometries (USD)
if 'asset_ead' not in globals():
    raise ValueError('Run EAD computation cells first so asset_ead exists in memory.')

network_map_details = network_details[[
    'sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column'
]].drop_duplicates().copy()

map_layers = []
missing_split_files = []

for row in network_map_details.itertuples(index=False):
    split_file = intersections_path / f"{row.asset_gpkg}_splits__coastal_flood_rasters_for_intersections__{row.asset_layer}.geoparquet"
    if not split_file.exists():
        missing_split_files.append(str(split_file))
        continue

    split_geom = geopandas.read_parquet(split_file)
    if split_geom.crs is not None:
        split_geom = split_geom.to_crs(jamaica_metric_grid_crs)
    if row.asset_id_column not in split_geom.columns:
        print(f"Skipping {row.asset_gpkg}_{row.asset_layer}: id column '{row.asset_id_column}' not in split file")
        continue

    split_geom = split_geom[[row.asset_id_column, 'geometry']].copy()
    split_geom = geopandas.GeoDataFrame(split_geom, geometry='geometry', crs=split_geom.crs)

    ead_subset = asset_ead.loc[
        (asset_ead['Asset'] == row.asset_gpkg) & (asset_ead['Layer'] == row.asset_layer),
        ['Asset_ID', 'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD']
    ].copy()

    if ead_subset.empty:
        continue

    split_geom['_join_id'] = split_geom[row.asset_id_column].astype(str)
    ead_subset['_join_id'] = ead_subset['Asset_ID'].astype(str)

    merged = split_geom.merge(
        ead_subset[['_join_id', 'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD']],
        on='_join_id',
        how='left'
    )

    merged['Sector'] = row.sector
    merged['Subsector'] = row.asset_description
    merged['Asset'] = row.asset_gpkg
    merged['Layer'] = row.asset_layer
    merged['Asset_ID'] = merged[row.asset_id_column]
    merged['Avoided_EAD_USD'] = merged['Avoided_EAD_USD'].fillna(0.0)

    map_layers.append(merged[[
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
        'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'geometry'
    ]])

if not map_layers:
    raise ValueError('No map layers could be built from split files.')

sector_avoided_ead_map_layers = geopandas.GeoDataFrame(
    pandas.concat(map_layers, ignore_index=True),
    geometry='geometry',
    crs=jamaica_metric_grid_crs
)

print(f"Map features loaded: {len(sector_avoided_ead_map_layers):,}")
print('Features by sector:')
display(sector_avoided_ead_map_layers.groupby('Sector', as_index=False).size())

if missing_split_files:
    print('Missing split files (skipped):')
    for p in sorted(set(missing_split_files)):
        print('-', p)



In [ ]:
# Plot avoided EAD map for all sectors combined with shared scale + explicit outlier highlights

value_col = avoided_ead_column
display_quantile = 0.995
cmap = avoided_ead_colormap

# Outlier colors by sign (important in both directions)
pos_outlier_color = '#145a32'  # avoided (positive)
neg_outlier_color = '#8b1a1a'  # increase (negative)

all_gdf = sector_avoided_ead_map_layers.copy()
raw_vals_usd = all_gdf[value_col].fillna(0.0)

# Use robust clipping for clearer contrast in all-sector view
abs_vals_usd = raw_vals_usd.abs()
global_abs_max_usd = float(abs_vals_usd.quantile(display_quantile))
if global_abs_max_usd <= 0:
    global_abs_max_usd = float(abs_vals_usd.max()) if float(abs_vals_usd.max()) > 0 else 1.0

# Choose readable units based on actual scale
if global_abs_max_usd >= 1e6:
    unit_factor = 1e6
    unit_label = 'USD millions'
elif global_abs_max_usd >= 1e3:
    unit_factor = 1e3
    unit_label = 'USD thousands'
else:
    unit_factor = 1.0
    unit_label = 'USD'

all_gdf['_plot_val'] = (raw_vals_usd / unit_factor).clip(
    -(global_abs_max_usd / unit_factor),
    +(global_abs_max_usd / unit_factor),
)
all_gdf['_is_outlier'] = raw_vals_usd.abs() > global_abs_max_usd
all_gdf['_outlier_sign'] = numpy.where(raw_vals_usd > 0, 'positive', numpy.where(raw_vals_usd < 0, 'negative', 'zero'))

global_abs_max = global_abs_max_usd / unit_factor

norm = TwoSlopeNorm(vmin=-global_abs_max, vcenter=0.0, vmax=global_abs_max)
print(
    f'All-sectors shared scale ({unit_label}, q={display_quantile:.3f}): '
    f'{-global_abs_max:,.2f} to +{global_abs_max:,.2f} (center 0)'
)
print(f"Outliers (|value| > cap): {int(all_gdf['_is_outlier'].sum()):,}")

fig, ax = plt.subplots(figsize=(10.8, 10.2))
ax.set_facecolor('#ffffff')
jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

geom_type = all_gdf.geometry.geom_type.astype(str)
polys = all_gdf[geom_type.str.contains('Polygon', na=False)]
lines = all_gdf[geom_type.str.contains('LineString', na=False)]
points = all_gdf[geom_type.str.contains('Point', na=False)]

if not polys.empty:
    polys.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.12, edgecolor='none', alpha=0.9, zorder=2)
if not lines.empty:
    lines.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.9, alpha=0.95, zorder=3)
if not points.empty:
    points.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, markersize=16, alpha=0.95, zorder=4)

# Explicitly highlight outliers, separated by sign
outliers = all_gdf[all_gdf['_is_outlier']].copy()
pos_outliers = outliers[outliers[value_col] > 0].copy()
neg_outliers = outliers[outliers[value_col] < 0].copy()

for subset, color in [(pos_outliers, pos_outlier_color), (neg_outliers, neg_outlier_color)]:
    if subset.empty:
        continue
    subset_geom_type = subset.geometry.geom_type.astype(str)
    subset_polys = subset[subset_geom_type.str.contains('Polygon', na=False)]
    subset_lines = subset[subset_geom_type.str.contains('LineString', na=False)]
    subset_points = subset[subset_geom_type.str.contains('Point', na=False)]

    if not subset_polys.empty:
        subset_polys.boundary.plot(ax=ax, color=color, linewidth=1.2, alpha=0.96, zorder=5)
    if not subset_lines.empty:
        subset_lines.plot(ax=ax, color=color, linewidth=2.0, alpha=0.96, zorder=5)
    if not subset_points.empty:
        subset_points.plot(ax=ax, color=color, markersize=38, alpha=0.96, zorder=5)

# Label largest outliers by absolute magnitude (both positive and negative)
label_n_each_sign = 6
top_pos = pos_outliers.assign(_abs_val=pos_outliers[value_col].abs()).sort_values('_abs_val', ascending=False).head(label_n_each_sign).copy()
top_neg = neg_outliers.assign(_abs_val=neg_outliers[value_col].abs()).sort_values('_abs_val', ascending=False).head(label_n_each_sign).copy()
label_df = pandas.concat([top_pos, top_neg], ignore_index=True)

if not label_df.empty:
    label_points = label_df.geometry.representative_point()
    for (_, row), pt in zip(label_df.iterrows(), label_points):
        v_scaled = row[value_col] / unit_factor
        is_pos = row[value_col] > 0
        label_color = pos_outlier_color if is_pos else neg_outlier_color
        ax.text(
            pt.x,
            pt.y,
            f"{v_scaled:+,.1f}",
            fontsize=7,
            color=label_color,
            ha='left',
            va='bottom',
            zorder=6,
            bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': label_color, 'pad': 0.35}
        )

# Save outlier table for auditing / finding exact assets
if not outliers.empty:
    outlier_table = outliers.assign(_abs_val=outliers[value_col].abs()).sort_values('_abs_val', ascending=False).copy()
    rp = outlier_table.geometry.representative_point()
    outlier_table['label_x'] = rp.x
    outlier_table['label_y'] = rp.y
    outlier_table['Outlier_Sign'] = numpy.where(outlier_table[value_col] > 0, 'avoided_positive', 'increase_negative')

    keep_cols = [
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', value_col,
        'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'Outlier_Sign', 'label_x', 'label_y'
    ]
    outlier_csv = map_out_dir / f"avoided_ead_outliers_all_sectors_usd_q{int(display_quantile * 1000)}.csv"
    outlier_table[keep_cols].to_csv(outlier_csv, index=False)
    print(f'Saved outlier table: {outlier_csv}')

# Keep map tightly framed to Jamaica so whitespace is minimal
minx, miny, maxx, maxy = jamaica_boundary.total_bounds
pad_x = (maxx - minx) * 0.01
pad_y = (maxy - miny) * 0.01
ax.set_xlim(minx - pad_x, maxx + pad_x)
ax.set_ylim(miny - pad_y, maxy + pad_y)

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

# Append colorbar axis directly under map to avoid large vertical gap
divider = make_axes_locatable(ax)
cax = divider.append_axes('bottom', size='3.2%', pad=0.08)
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
cbar.set_label(
    f'Avoided EAD ({unit_label}), clipped at q={display_quantile:.3f} | '
    'Red=increase, White=no change, Green=avoided'
)

ax.set_title(
    'Total avoided EAD locations across all sectors',
    fontsize=13,
)
ax.set_axis_off()
fig.subplots_adjust(top=0.94)

out_png = map_out_dir / 'avoided_ead_map_all_sectors_usd_shared_scale.png'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
print(f'Saved: {out_png}')
plt.show()



In [ ]:
# Plot avoided EAD maps for each sector with a shared global scale (USD)

value_col = avoided_ead_column
cmap = avoided_ead_colormap

# Shared global scale using robust percentile to avoid all-yellow maps
all_vals = sector_avoided_ead_map_layers[value_col].fillna(0.0)
abs_vals = all_vals.abs()
global_abs_max = float(abs_vals.quantile(0.995))
if global_abs_max <= 0:
    global_abs_max = float(abs_vals.max()) if float(abs_vals.max()) > 0 else 1.0

norm = TwoSlopeNorm(vmin=-global_abs_max, vcenter=0.0, vmax=global_abs_max)
print(f'Global shared USD scale (robust): {-global_abs_max:,.2f} to +{global_abs_max:,.2f} (center 0)')

for sector_name in sector_order:
    sector_gdf = sector_avoided_ead_map_layers[
        sector_avoided_ead_map_layers['Sector'] == sector_name
    ].copy()

    if sector_gdf.empty:
        print(f'Skipping {sector_name}: no features')
        continue

    # Clip for plotting so outliers do not flatten color contrast
    sector_gdf['_plot_val'] = sector_gdf[value_col].clip(-global_abs_max, global_abs_max)

    fig, ax = plt.subplots(figsize=(10.5, 9.0))
    ax.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

    geom_type = sector_gdf.geometry.geom_type.astype(str)
    polys = sector_gdf[geom_type.str.contains('Polygon', na=False)]
    lines = sector_gdf[geom_type.str.contains('LineString', na=False)]
    points = sector_gdf[geom_type.str.contains('Point', na=False)]

    if not polys.empty:
        polys.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.12, edgecolor='none', alpha=0.9, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.9, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, markersize=16, alpha=0.95, zorder=4)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label('Avoided EAD (USD) | Green = avoided, Red = increase', rotation=90)

    ax.set_title(f"Avoided EAD map - {sector_name.capitalize()} (USD, shared scale)", fontsize=13)
    ax.set_axis_off()
    plt.tight_layout()

    out_png = map_out_dir / f"avoided_ead_map_{sector_name}_usd_shared_scale.png"
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    print(f'Saved: {out_png}')
    plt.show()



In [ ]:
# Plot avoided EAD maps for each sector with sector-specific accurate scales (USD)

value_col = avoided_ead_column
cmap = avoided_ead_colormap

for sector_name in sector_order:
    sector_gdf = sector_avoided_ead_map_layers[
        sector_avoided_ead_map_layers['Sector'] == sector_name
    ].copy()

    if sector_gdf.empty:
        print(f'Skipping {sector_name}: no features')
        continue

    vals = sector_gdf[value_col].fillna(0.0)
    sector_abs_max = float(vals.abs().max())
    if sector_abs_max <= 0:
        sector_abs_max = 1.0

    norm = TwoSlopeNorm(vmin=-sector_abs_max, vcenter=0.0, vmax=sector_abs_max)
    print(f"{sector_name.capitalize()} scale (USD): {-sector_abs_max:,.2f} to +{sector_abs_max:,.2f}")

    fig, ax = plt.subplots(figsize=(10.5, 9.0))
    ax.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=ax, color='#c7c7c7', linewidth=0.4, zorder=1)

    geom_type = sector_gdf.geometry.geom_type.astype(str)
    polys = sector_gdf[geom_type.str.contains('Polygon', na=False)]
    lines = sector_gdf[geom_type.str.contains('LineString', na=False)]
    points = sector_gdf[geom_type.str.contains('Point', na=False)]

    if not polys.empty:
        polys.plot(ax=ax, column=value_col, cmap=cmap, norm=norm, linewidth=0.1, edgecolor='none', alpha=0.92, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column=value_col, cmap=cmap, norm=norm, linewidth=1.0, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column=value_col, cmap=cmap, norm=norm, markersize=18, alpha=0.95, zorder=4)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label('Avoided EAD (USD) | Red = increase, White = no change, Green = avoided', rotation=90)

    ax.set_title(f"Avoided EAD map - {sector_name.capitalize()} (USD, sector-specific scale)", fontsize=13)
    ax.set_axis_off()
    plt.tight_layout()

    out_png = map_out_dir / f"avoided_ead_map_{sector_name}_usd_sector_scale.png"
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    print(f'Saved: {out_png}')
    plt.show()



In [ ]:
# Plot USD avoided EAD maps with percentile clipping + explicit outlier highlighting

value_col = avoided_ead_column
display_quantile = 0.995
outlier_highlight_color = '#145a32'
cmap = avoided_ead_colormap

for sector_name in sector_order:
    sector_gdf = sector_avoided_ead_map_layers[
        sector_avoided_ead_map_layers['Sector'] == sector_name
    ].copy()

    if sector_gdf.empty:
        print(f'Skipping {sector_name}: no features')
        continue

    vals = sector_gdf[value_col].fillna(0.0)
    abs_vals = vals.abs()

    sector_abs_max = float(abs_vals.max())
    display_cap = float(abs_vals.quantile(display_quantile))

    if display_cap <= 0:
        display_cap = sector_abs_max if sector_abs_max > 0 else 1.0
    if sector_abs_max <= 0:
        sector_abs_max = display_cap

    sector_gdf['_plot_val'] = vals.clip(-display_cap, display_cap)
    sector_gdf['_is_outlier'] = abs_vals > display_cap

    norm = TwoSlopeNorm(vmin=-display_cap, vcenter=0.0, vmax=display_cap)

    n_outliers = int(sector_gdf['_is_outlier'].sum())
    print(
        f"{sector_name.capitalize()}: display scale +/-{display_cap:,.2f} USD "
        f"(q={display_quantile:.3f}), true max abs={sector_abs_max:,.2f} USD, outliers={n_outliers}"
    )

    fig, ax = plt.subplots(figsize=(10.5, 9.0))
    ax.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=ax, color='#c7c7c7', linewidth=0.35, zorder=1)

    geom_type = sector_gdf.geometry.geom_type.astype(str)
    polys = sector_gdf[geom_type.str.contains('Polygon', na=False)]
    lines = sector_gdf[geom_type.str.contains('LineString', na=False)]
    points = sector_gdf[geom_type.str.contains('Point', na=False)]

    if not polys.empty:
        polys.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.08, edgecolor='none', alpha=0.92, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.95, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, markersize=17, alpha=0.95, zorder=4)

    outliers = sector_gdf[sector_gdf['_is_outlier']].copy()
    if not outliers.empty:
        out_geom_type = outliers.geometry.geom_type.astype(str)
        out_polys = outliers[out_geom_type.str.contains('Polygon', na=False)]
        out_lines = outliers[out_geom_type.str.contains('LineString', na=False)]
        out_points = outliers[out_geom_type.str.contains('Point', na=False)]

        if not out_polys.empty:
            out_polys.boundary.plot(ax=ax, color=outlier_highlight_color, linewidth=1.2, alpha=0.95, zorder=5)
        if not out_lines.empty:
            out_lines.plot(ax=ax, color=outlier_highlight_color, linewidth=2.1, alpha=0.95, zorder=5)
        if not out_points.empty:
            out_points.plot(ax=ax, color=outlier_highlight_color, markersize=36, alpha=0.95, zorder=5)

        # Label the largest outliers so they are easy to find in-map
        top_outliers = outliers.assign(_abs_val=outliers[value_col].abs()).sort_values('_abs_val', ascending=False).head(8).copy()
        label_points = top_outliers.geometry.representative_point()
        for (_, row), pt in zip(top_outliers.iterrows(), label_points):
            ax.text(
                pt.x,
                pt.y,
                f"{row[value_col]:,.0f}",
                fontsize=7,
                color=outlier_highlight_color,
                ha='left',
                va='bottom',
                zorder=6,
                bbox={'facecolor': 'white', 'alpha': 0.75, 'edgecolor': outlier_highlight_color, 'pad': 0.4}
            )

        # Save a table of outliers for auditing / finding exact assets
        outlier_table = outliers.assign(_abs_val=outliers[value_col].abs()).sort_values('_abs_val', ascending=False).copy()
        rp = outlier_table.geometry.representative_point()
        outlier_table['label_x'] = rp.x
        outlier_table['label_y'] = rp.y
        keep_cols = [
            'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', value_col,
            'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'label_x', 'label_y'
        ]
        outlier_csv = map_out_dir / f"avoided_ead_outliers_{sector_name}_usd_q{int(display_quantile * 1000)}.csv"
        outlier_table[keep_cols].to_csv(outlier_csv, index=False)
        print(f'Saved outlier table: {outlier_csv}')

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label(
        f"Avoided EAD (USD), clipped at q={display_quantile:.3f} | Red=increase, White=no change, Green=avoided",
        rotation=90,
    )

    ax.set_title(
        f"Avoided EAD map - {sector_name.capitalize()} (USD, clipped display + outlier highlights)",
        fontsize=12,
    )
    ax.set_axis_off()
    plt.tight_layout()

    out_png = map_out_dir / f"avoided_ead_map_{sector_name}_usd_q{int(display_quantile * 1000)}_outliers.png"
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    print(f'Saved map: {out_png}')
    plt.show()

